# X-OPT - FACILITY REMOVAL REOPTIMIZATION WITH MAX-K-CUT VARIANTS

Per instance, the exact optimum is used as the reference solution. For each selected source instance, the notebook generates one reoptimization scenario per exact facility, capped at `MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE`. In each scenario, one facility from that solution is removed, the remaining optimal facilities are fixed, and each variant defines the replacement candidate pool.

**Variants**:

1. `baseline_all_facilities`: all non-forbidden facilities are available;
2. `max_k_cut_original`: original max-k-cut on the top-fraction LTM;
3. `max_k_cut_mst_trajectory`: connect disconnected top-fraction LTM components with the existing `lib` trajectory completion routine, then apply max-k-cut;
4. `max_k_cut_pairwise_trajectory`: connect every pair of disconnected components with trajectories, then apply max-k-cut;
5. `max_k_cut_mst_range_guard`: use the existing trajectory completion routine, but if generated trajectory solutions leave the full LTM cost interval before `TOP_FRACTION`, discard the trajectory additions and use only the component containing the metaheuristic best solution before applying max-k-cut;
6. `max_k_cut_pairwise_range_guard`: connect every pair of disconnected components with trajectories, but apply the same full-LTM cost range guard before max-k-cut;
7. `random_original_pool_size`: choose the same number of replacement candidates as `max_k_cut_original`, sampling randomly after excluding the metaheuristic best solution;
8. `nearest_original_pool_size`: choose the same number of replacement candidates as `max_k_cut_original`, taking the facilities closest to the removed facility after excluding the metaheuristic best solution.

> Obs.: This notebook extends `5.0 Exact Facility Removal Reoptimization.ipynb` to compare eight reoptimization variants on the first 20 OR-Library p-median instances.

### SETUP

In [1]:
from __future__ import annotations

import gc
import os
import sys
import random

import numpy    as np
import pandas   as pd
import networkx as nx

from tqdm.auto       import tqdm
from pathlib         import Path
from IPython.display import display
from time            import perf_counter
from mip             import BINARY, Model, OptimizationStatus, minimize, xsum


pd.set_option("display.max_columns" , None)
pd.set_option("display.max_colwidth", 120 )

In [2]:
from lib.paths import (
    find_project_root      ,
    instance_sort_key      ,
    canonical_instance_name,
)
from lib.instances import (
    list_orlibrary_instances        ,
    apply_instance_selection        ,
    read_instance_metadata          ,
    load_best_known_costs_to_dict_id,
)

from lib.graph import (
    read_orlibrary_graph     ,
    all_pairs_shortest_paths ,
    build_top_ltm            ,
    build_cooccurrence_matrix,
    total_edge_count         ,
    total_edge_weight        ,
)
from lib.ltm import (
    binary_facility_vector                          ,
    facilities_from_binary_vector                   ,
    pmedian_solution_cost                           ,
    build_solution_swap_graph                       ,
    build_swap_component_graph                      ,
    greedy_swap_path_between_solutions              ,
    complete_long_term_memory_with_swap_trajectories,
)

from lib.maxcut import (
    max_p_cut_local_search  ,
    best_facility_separation,
)

from lib.explain import extract_partition_groups
from lib.mip     import extract_open_facilities_candidates
from lib.metrics import compute_solution_cost, gap_to_reference_percent

from lib.utils   import (
    finite_or_none          ,
    parse_optional_int_env  ,
    parse_optional_float_env,
    as_sorted_tuple         ,
    format_facilities       ,
    format_groups           ,
)

In [3]:
PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print(f"Project root is {PROJECT_ROOT}.")

Project root is /home/rei-luisinho/xopt.


In [4]:
import pymedian

### CONFIGURATION

In [5]:
def first_optional_int_env(
    *names: str, default: int | None = None
) -> int | None:
    for name in names:
        value = parse_optional_int_env(name)

        if value is not None:
            return value

    return default


def first_optional_float_env(
    *names: str, default: float | None = None
) -> float | None:
    for name in names:
        value = parse_optional_float_env(name)

        if value is not None:
            return value

    return default

In [6]:
INSTANCES_DIR = PROJECT_ROOT  / "instances"
PMEDOPT_PATH  = INSTANCES_DIR / "pmedopt.txt"
OUTPUT_DIR    = PROJECT_ROOT  / "notebooks" / "experiments_sbpo" / "artifacts"

RAW_RESULTS_CSV = OUTPUT_DIR / "facility_removal_connected_max_k_cut_raw.csv"
STRUCTURES_CSV  = OUTPUT_DIR / "facility_removal_connected_max_k_cut_structures.csv"
SUMMARY_CSV     = OUTPUT_DIR / "facility_removal_connected_max_k_cut_summary.csv"
FAILURES_CSV    = OUTPUT_DIR / "facility_removal_connected_max_k_cut_failures.csv"


print(f"Raw results CSV: {RAW_RESULTS_CSV}")

Raw results CSV: /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_connected_max_k_cut_raw.csv


In [7]:
INSTANCE_FILTER = os.getenv("FACREM_CONN_INSTANCE_FILTER") or os.getenv("FACREM_INSTANCE_FILTER")
DETAILS_FORMAT  = os.getenv("FACREM_CONN_DETAILS_FORMAT" ) or os.getenv("FACREM_DETAILS_FORMAT", "binary")

MAX_INSTANCES = first_optional_int_env  ("FACREM_CONN_MAX_INSTANCES", "FACREM_MAX_INSTANCES", default=20  )
RESTARTS      = first_optional_int_env  ("FACREM_CONN_META_RESTARTS", "FACREM_META_RESTARTS", default=8   )
MAX_ITER      = first_optional_int_env  ("FACREM_CONN_META_MAX_ITER", "FACREM_META_MAX_ITER", default=25  )
FACTOR        = first_optional_int_env  ("FACREM_CONN_META_FACTOR"  , "FACREM_META_FACTOR"  , default=1   )
TOP_FRACTION  = first_optional_float_env("FACREM_CONN_TOP_FRACTION" , "FACREM_TOP_FRACTION" , default=0.10)

EXACT_TIME_LIMIT_SECONDS = first_optional_int_env("FACREM_CONN_EXACT_TIME_LIMIT_SECONDS", "FACREM_EXACT_TIME_LIMIT_SECONDS", default=400)
REOPT_TIME_LIMIT_SECONDS = first_optional_int_env("FACREM_CONN_REOPT_TIME_LIMIT_SECONDS", "FACREM_REOPT_TIME_LIMIT_SECONDS", default=180)

GLOBAL_SEED        = first_optional_int_env("FACREM_CONN_GLOBAL_SEED"       , "FACREM_GLOBAL_SEED"       , default=42  )
MAX_K_CUT_RESTARTS = first_optional_int_env("FACREM_CONN_MAX_K_CUT_RESTARTS", "FACREM_MAX_P_CUT_RESTARTS", default=10  )
MAX_K_CUT_MAX_ITER = first_optional_int_env("FACREM_CONN_MAX_K_CUT_MAX_ITER", "FACREM_MAX_P_CUT_MAX_ITER", default=1000)


if DETAILS_FORMAT != "binary":
    raise ValueError("This notebook expects DETAILS_FORMAT='binary'.")


print(f"Exact model time limit (s)    : {EXACT_TIME_LIMIT_SECONDS}")
print(f"Reoptimization time limit (s) : {REOPT_TIME_LIMIT_SECONDS}")
print(f"Top fraction kept from LTM    : {        TOP_FRACTION:.0%}")
print(f"Metaheuristic parameters      : restarts={          RESTARTS}, max_iter={          MAX_ITER}, factor={   FACTOR}")
print(f"Max-k-cut parameters          : restarts={MAX_K_CUT_RESTARTS}, max_iter={MAX_K_CUT_MAX_ITER}, seed={GLOBAL_SEED}")

Exact model time limit (s)    : 400
Reoptimization time limit (s) : 180
Top fraction kept from LTM    : 10%
Metaheuristic parameters      : restarts=8, max_iter=25, factor=1
Max-k-cut parameters          : restarts=10, max_iter=1000, seed=42


In [8]:
_REQUESTED_MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE = first_optional_int_env(
    "FACREM_CONN_MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE",
    "FACREM_MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE"     ,
    default=3,
)

if _REQUESTED_MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE is None:
    _REQUESTED_MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE = 3
if _REQUESTED_MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE < 1:
    raise ValueError("The reoptimization scenario limit must be positive.")

MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE = min(5, _REQUESTED_MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE)

SAVE_RESULTS_CSV = os.getenv("FACREM_CONN_SAVE_RESULTS_CSV", os.getenv("FACREM_SAVE_RESULTS_CSV", "1")) != "0"

In [9]:
ALL_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES     = apply_instance_selection(
    ALL_INSTANCE_NAMES,
    pattern=INSTANCE_FILTER,
    limit  =MAX_INSTANCES  ,
)

BEST_KNOWN_COSTS = load_best_known_costs_to_dict_id(PMEDOPT_PATH)


if not INSTANCE_NAMES:
    raise FileNotFoundError(f"No instances were selected from {INSTANCES_DIR}.")


print(f"Instances discovered : {len(ALL_INSTANCE_NAMES)}")
print(f"Instances selected   : {len( INSTANCE_NAMES   )}")
print(f"First selected       : {     INSTANCE_NAMES[:5]}")

expected_reoptimization_scenarios = sum(
    min(
        int(read_instance_metadata(INSTANCES_DIR / name).get("p", 0) or 0),
        MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE                         ,
    )
    for name in INSTANCE_NAMES
)

print()
print(f"Max reoptimization scenarios per instance : {MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE}")
print(f"Expected reoptimization scenarios         : {        expected_reoptimization_scenarios}")

Instances discovered : 40
Instances selected   : 20
First selected       : ['pmed1.txt', 'pmed2.txt', 'pmed3.txt', 'pmed4.txt', 'pmed5.txt']

Max reoptimization scenarios per instance : 3
Expected reoptimization scenarios         : 60


### MEMORY AND MAX-K-CUT HELPERS

In [10]:
def best_facilities_from_summary(
    summary: dict[str, object]
) -> tuple[int, ...]:
    return tuple(
        sorted(
            int(value) - 1
            for value in summary["tspmed_facilities"]
        )
    )


def ltm_cost_range(
    long_term_memory: list[dict[str, object]]
) -> tuple[float, float]:
    costs = [
        float(record["cost"])
        for record in long_term_memory
        if  record.get("cost") is not None
    ]

    if not costs:
        return np.nan, np.nan

    return float(min(costs)), float(max(costs))


def ltm_records_to_solutions(
    long_term_memory : list[dict[str, object]],
    *,
    facilities_key   : str = "facilities",
) -> list[tuple[int, ...]]:
    return [
        facilities_from_binary_vector(
            record[facilities_key]
        )
        for record in long_term_memory
    ]


def record_cost_or_compute(
    record   : dict [str, object],
    solution : tuple[int, ...   ],
    cost_fn
) -> float:
    cost = record.get("cost")

    if cost is None or pd.isna(cost):
        return float(cost_fn(solution))

    return float(cost)


def summarize_ltm_connectivity(
    long_term_memory: list[dict[str, object]]
) -> dict[str, object]:
    if not long_term_memory:
        return {
            "solution_count"          : 0   ,
            "swap_edges"              : 0   ,
            "component_count"         : 0   ,
            "largest_component_size"  : 0   ,
            "smallest_component_size" : 0   ,
            "is_connected"            : True,
        }

    solutions = ltm_records_to_solutions(long_term_memory)
    costs     = [
        float(record.get("cost", np.nan))
        for record in long_term_memory
    ]

    graph      = build_solution_swap_graph(solutions, costs)
    components = [sorted(component) for component in nx.connected_components(graph)]
    sizes      = [len   (component) for component in components                    ]

    return {
        "solution_count"  : len(solutions ),
        "component_count" : len(components),
        "swap_edges"      : graph.number_of_edges(),

        "largest_component_size"  : max(sizes) if sizes else 0,
        "smallest_component_size" : min(sizes) if sizes else 0,

        "is_connected" : graph.number_of_nodes() <= 1 or nx.is_connected(graph),
    }


def select_best_solution_component(
    long_term_memory : list[dict[str, object]],
    best_facilities  : tuple[int, ...] | list[int],
) -> tuple[list[dict[str, object]], dict[str, object]]:
    if not long_term_memory:
        raise ValueError("Long term memory is empty.")

    best_facilities = as_sorted_tuple         (best_facilities )
    solutions       = ltm_records_to_solutions(long_term_memory)
    costs           = np.asarray(
        [
            float(record.get("cost", np.nan))
            for record in long_term_memory
        ],
        dtype=float
    )

    try:
        best_index = solutions.index(best_facilities)
        reason     = "exact_best_solution_match"
    except ValueError:
        best_index = int(np.nanargmin(costs))
        reason     = "minimum_cost_record"

    graph            = build_solution_swap_graph  (solutions, costs.tolist())
    component_nodes  = sorted(nx.node_connected_component(graph, best_index))

    selected_records = [
        dict(long_term_memory[index])
        for index in component_nodes
    ]

    return selected_records, {
        "best_component_solution_count"   : len(component_nodes),
        "best_component_selected_index"   : best_index,
        "best_component_selection_reason" : reason    ,
    }

In [11]:
def complete_long_term_memory_pairwise_swap_trajectories(
    instance         : object                 ,
    long_term_memory : list[dict[str, object]],
    *,
    cost_fn=None,
) -> dict[str, object]:
    if not long_term_memory:
        raise ValueError("Long term memory is empty.")

    n = int(
        np.asarray(long_term_memory[0]["facilities"], dtype=np.int8).size
    )

    effective_cost_fn = cost_fn                  \
                        if   cost_fn is not None \
                        else lambda solution: pmedian_solution_cost(instance, solution)

    original_records  = [
        dict(record)
        for record in long_term_memory
    ]

    original_solutions = ltm_records_to_solutions(original_records)
    original_costs     = [
        record_cost_or_compute(record, solution, effective_cost_fn)
        for record, solution in zip(original_records, original_solutions)
    ]

    for record, cost in zip(original_records, original_costs):
        vector = np.asarray(record["facilities"], dtype=np.int8)

        if vector.ndim != 1 or int(vector.size) != n:
            raise ValueError("All facility vectors must be one-dimensional and match instance size.")

        record["cost"] = float(cost)

    original_graph = build_solution_swap_graph(original_solutions, original_costs)

    original_is_connected       = original_graph.number_of_nodes() <= 1 or nx.is_connected(original_graph)
    component_graph, components = build_swap_component_graph(original_graph, original_solutions)

    completed_records   = list(original_records  )
    completed_solutions = list(original_solutions)

    solution_origin   = ["original"] * len(completed_solutions)
    solution_to_index = {
        solution : index
        for index, solution in enumerate(completed_solutions)
    }
    trajectory_rows   = []

    if not original_is_connected:
        for source_component, target_component, edge_data in component_graph.edges(data=True):
            source_solution = original_solutions[int(edge_data["source_solution"])]
            target_solution = original_solutions[int(edge_data["target_solution"])]
            path, path_rows = greedy_swap_path_between_solutions(
                source_solution,
                target_solution,
                cost_fn=effective_cost_fn,
            )

            previous_solution = path[0]
            previous_index    = solution_to_index[previous_solution]

            for path_position, next_solution in enumerate(path[1:], start=1):
                if next_solution not in solution_to_index:
                    solution_to_index[next_solution] = len(completed_solutions)

                    completed_solutions.append(next_solution)
                    completed_records  .append(
                        {
                            "cost"       : float(effective_cost_fn(next_solution))          ,
                            "facilities" : binary_facility_vector(n, next_solution).tolist(),
                        }
                    )
                    solution_origin    .append("trajectory")

                next_index = solution_to_index[next_solution    ]
                step_data  = path_rows        [path_position - 1]

                trajectory_rows.append(
                    {
                        "source_component" : int(source_component),
                        "target_component" : int(target_component),

                        "from_solution" : previous_index + 1,
                        "to_solution"   : next_index     + 1,

                        "removed"               : step_data["removed"],
                        "added"                 : step_data["added"  ],
                        "cost"                  : step_data["cost"   ],
                        "swap_distance_to_goal" : step_data["swap_distance_left"],
                        "solution"              : step_data["solution"          ],
                    }
                )

                previous_solution = next_solution
                previous_index    = next_index

    completed_costs = [
        record_cost_or_compute(record, solution, effective_cost_fn)
        for record, solution in zip(completed_records, completed_solutions)
    ]

    completed_graph        = build_solution_swap_graph(completed_solutions, completed_costs)
    completed_is_connected = completed_graph.number_of_nodes() <= 1 or nx.is_connected(completed_graph)

    return {
        "long_term_memory": completed_records  ,
        "solutions"       : completed_solutions,
        "solution_origin" : solution_origin    ,
        "trajectory_rows" : trajectory_rows    ,

        "original_swap_graph"  : original_graph ,
        "completed_swap_graph" : completed_graph,

        "component_graph"           : component_graph       ,
        "components"                : components            ,
        "original_is_connected"     : original_is_connected ,
        "completed_is_connected"    : completed_is_connected,
        "original_component_count"  :                            len(components     ),
        "completed_component_count" : nx.number_connected_components(completed_graph),

        "original_solution_count"  : len(original_records ),
        "completed_solution_count" : len(completed_records),
        "added_solution_count"     : len(completed_records) - len(original_records),

        "movement_count"       : len(trajectory_rows)             ,
        "component_edge_count" : component_graph.number_of_edges(),
    }


def apply_long_term_memory_range_guard(
    completion       : dict[str, object]        ,
    long_term_memory : list[dict[str, object]]  ,
    *,
    full_cost_range : tuple[float, float]        ,
    best_facilities : tuple[int, ...] | list[int],
) -> dict[str, object]:
    result = dict(completion)

    lower, upper = full_cost_range

    out_of_range = []
    for index, (record, origin) in enumerate(
        zip(
            result["long_term_memory"],
            result["solution_origin" ],
        )
    ):
        if origin != "trajectory":
            continue

        cost = float(record["cost"])
        if cost > upper + 1e-9:
            out_of_range.append(
                {
                    "record_index" : index,
                    "cost"         : cost ,
                }
            )

    result["guard_fallback_used"] = False
    result["best_component_solution_count"  ] = np.nan
    result["best_component_selection_reason"] = None
    result["trajectory_generated_out_of_range_count"] = len(out_of_range)
    result["trajectory_generated_out_of_range_costs"] = out_of_range

    if not out_of_range:
        return result

    selected_records, metadata = select_best_solution_component(
        long_term_memory, best_facilities
    )

    selected_solutions = ltm_records_to_solutions(selected_records)
    selected_costs     = [
        float(record["cost"])
        for record in selected_records
    ]
    selected_graph     = build_solution_swap_graph(selected_solutions, selected_costs)

    result.update(
        {
            "long_term_memory" : selected_records  ,
            "solutions"        : selected_solutions,

            "solution_origin" : ["best_component"] * len(selected_records),
            "trajectory_rows" : [],

            "completed_swap_graph"      : selected_graph,
            "completed_is_connected"    : selected_graph.number_of_nodes() <= 1 or nx.is_connected(selected_graph),
            "completed_component_count" : nx.number_connected_components(selected_graph  ),
            "completed_solution_count"  :                          len  (selected_records),

            "added_solution_count" : 0,
            "movement_count"       : 0,
            "guard_fallback_used"  : True,

            **metadata,
        }
    )

    return result


def complete_long_term_memory_mst_with_range_guard(
    instance         : object                 ,
    long_term_memory : list[dict[str, object]],
    *,
    full_cost_range : tuple[float, float]        ,
    best_facilities : tuple[int, ...] | list[int],
    cost_fn=None,
) -> dict[str, object]:
    result = complete_long_term_memory_with_swap_trajectories(
        instance        ,
        long_term_memory,
        cost_fn=cost_fn ,
    )

    return apply_long_term_memory_range_guard(
        result          ,
        long_term_memory,
        full_cost_range=full_cost_range,
        best_facilities=best_facilities ,
    )


def complete_long_term_memory_pairwise_with_range_guard(
    instance         : object                 ,
    long_term_memory : list[dict[str, object]],
    *,
    full_cost_range : tuple[float, float]        ,
    best_facilities : tuple[int, ...] | list[int],
    cost_fn=None,
) -> dict[str, object]:
    result = complete_long_term_memory_pairwise_swap_trajectories(
        instance        ,
        long_term_memory,
        cost_fn=cost_fn ,
    )

    return apply_long_term_memory_range_guard(
        result          ,
        long_term_memory,
        full_cost_range=full_cost_range,
        best_facilities=best_facilities ,
    )

In [12]:
def completion_edge_count(completion: dict[str, object] | None) -> float:
    if completion is None:
        return np.nan

    if "component_edge_count" in completion:
        return float(completion["component_edge_count"])

    component_mst = completion.get("component_mst")
    if component_mst is not None:
        return float(component_mst.number_of_edges())

    component_graph = completion.get("component_graph")
    if component_graph is not None:
        return float(component_graph.number_of_edges())

    return np.nan


def extract_max_k_cut_from_memory(
    summary          : dict[     str, object ],
    long_term_memory : list[dict[str, object]],
    *,
    memory_strategy : str,
    completion      : dict[str, object] | None = None,
) -> dict[str, object]:
    if not long_term_memory:
        raise ValueError("Long term memory is empty.")

    matrix    = np.vstack(
        [
            np.asarray(record["facilities"], dtype=np.int8)
            for record in long_term_memory
        ]
    )

    adjacency = build_cooccurrence_matrix(matrix)

    p = int(summary["p"])

    labels, cut_weight, internal_weight = max_p_cut_local_search(
        adjacency,
        p        ,
        n_restarts=MAX_K_CUT_RESTARTS,
        max_iter  =MAX_K_CUT_MAX_ITER,
        seed      =GLOBAL_SEED       ,
    )

    groups             = extract_partition_groups    (labels, p)
    best_facilities    = best_facilities_from_summary(summary  )
    connectivity       = summarize_ltm_connectivity(long_term_memory)
    cost_min, cost_max = ltm_cost_range            (long_term_memory)

    result = {
        "memory_strategy" : memory_strategy,

        "analysis_memory_size"     : len(long_term_memory),
        "analysis_memory_min_cost" : cost_min,
        "analysis_memory_max_cost" : cost_max,

        "analysis_solution_count"          : connectivity["solution_count"         ],
        "analysis_swap_edges"              : connectivity["swap_edges"             ],
        "analysis_component_count"         : connectivity["component_count"        ],
        "analysis_largest_component_size"  : connectivity["largest_component_size" ],
        "analysis_smallest_component_size" : connectivity["smallest_component_size"],
        "analysis_is_connected"            : connectivity["is_connected"           ],

        "cooccurrence_edges"        : total_edge_count (adjacency),
        "cooccurrence_total_weight" : total_edge_weight(adjacency),

        "max_k_cut_groups"                   : groups,
        "max_k_cut_fraction_cut"             : cut_weight / max(1e-12, cut_weight + internal_weight ),
        "max_k_cut_best_facility_separation" : best_facility_separation(labels, set(best_facilities)),

        "pre_connection_component_count"  : np.nan,
        "post_connection_component_count" : connectivity["component_count"],
        "pre_connection_solution_count"   : np.nan,
        "post_connection_solution_count"  : len(long_term_memory),

        "trajectory_added_solution_count"         : 0,
        "trajectory_movement_count"               : 0,
        "trajectory_connection_edge_count"        : np.nan,
        "trajectory_generated_out_of_range_count" : 0,

        "guard_fallback_used"             : False ,
        "best_component_solution_count"   : np.nan,
        "best_component_selection_reason" : None  ,
    }

    if completion is not None:
        result.update(
            {
                "pre_connection_component_count"  : completion.get("original_component_count" ),
                "post_connection_component_count" : completion.get("completed_component_count"),

                "pre_connection_solution_count"   : completion.get("original_solution_count" ),
                "post_connection_solution_count"  : completion.get("completed_solution_count"),
                "trajectory_added_solution_count" : completion.get("added_solution_count" , 0),
                "trajectory_movement_count"       : completion.get("movement_count"       , 0),

                "trajectory_connection_edge_count"        : completion_edge_count(completion),
                "trajectory_generated_out_of_range_count" : completion.get("trajectory_generated_out_of_range_count", 0),

                "guard_fallback_used" : bool(completion.get("guard_fallback_used", False)),

                "best_component_solution_count"   : completion.get("best_component_solution_count"  , np.nan),
                "best_component_selection_reason" : completion.get("best_component_selection_reason"),
            }
        )

    return result


def build_max_k_cut_memory_analyses(summary, details, distances):
    full_ltm = details["long_term_memory"]

    if not full_ltm:
        raise ValueError("Long term memory is empty.")

    top_ltm, _, top_costs = build_top_ltm(full_ltm, TOP_FRACTION)
    full_min, full_max = ltm_cost_range(full_ltm)
    top_min , top_max  = ltm_cost_range(top_ltm )

    base_metadata = {
        "source_ltm_size"    : len(full_ltm),
        "top_solution_count" : len(top_ltm ),
        "top_fraction"       : TOP_FRACTION ,
        "top_cost_cutoff"    : float(top_costs.max()),

        "full_ltm_cost_min" : full_min,
        "full_ltm_cost_max" : full_max,
        "top_ltm_cost_min"  : top_min ,
        "top_ltm_cost_max"  : top_max ,
    }

    cost_fn = lambda solution: compute_solution_cost(distances, solution)

    best_facilities = best_facilities_from_summary(summary)

    analyses = {}

    analyses["max_k_cut_original"] = extract_max_k_cut_from_memory(
        summary,
        top_ltm,
        memory_strategy="top_ltm_original",
    )

    mst_completion = complete_long_term_memory_with_swap_trajectories(
        distances,
        top_ltm  ,
        cost_fn=cost_fn,
    )
    analyses["max_k_cut_mst_trajectory"] = extract_max_k_cut_from_memory(
        summary,
        mst_completion["long_term_memory"],
        memory_strategy="top_ltm_mst_trajectory",
        completion     =mst_completion          ,
    )

    pairwise_completion = complete_long_term_memory_pairwise_swap_trajectories(
        distances,
        top_ltm  ,
        cost_fn=cost_fn,
    )
    analyses["max_k_cut_pairwise_trajectory"] = extract_max_k_cut_from_memory(
        summary,
        pairwise_completion["long_term_memory"],
        memory_strategy="top_ltm_pairwise_trajectory",
        completion     =pairwise_completion          ,
    )

    guarded_completion = complete_long_term_memory_mst_with_range_guard(
        distances,
        top_ltm  ,
        full_cost_range=(full_min, full_max),
        best_facilities=best_facilities     ,
        cost_fn=cost_fn,
    )
    analyses["max_k_cut_mst_range_guard"] = extract_max_k_cut_from_memory(
        summary,
        guarded_completion["long_term_memory"],
        memory_strategy="top_ltm_mst_trajectory_range_guard",
        completion     =guarded_completion                  ,
    )

    pairwise_guarded_completion = complete_long_term_memory_pairwise_with_range_guard(
        distances,
        top_ltm  ,
        full_cost_range=(full_min, full_max),
        best_facilities=best_facilities     ,
        cost_fn=cost_fn,
    )
    analyses["max_k_cut_pairwise_range_guard"] = extract_max_k_cut_from_memory(
        summary,
        pairwise_guarded_completion["long_term_memory"],
        memory_strategy="top_ltm_pairwise_trajectory_range_guard",
        completion     =pairwise_guarded_completion              ,
    )

    for analysis in analyses.values():
        analysis.update(base_metadata)

    return analyses, base_metadata

### EXACT MODEL AND REOPTIMIZATION

In [13]:
def reoptimization_removals(
    open_facilities            : tuple[int, ...] | list[int],
    max_scenarios_per_instance : int,
) -> tuple[tuple[int, int], ...]:
    facilities = as_sorted_tuple(open_facilities)

    if not facilities:
        raise ValueError("No facilities are available to remove.")

    limit = min(len(facilities), int(max_scenarios_per_instance), 10)
    if limit < 1:
        raise ValueError("At least one reoptimization scenario must be generated per instance.")

    rng = random.Random(GLOBAL_SEED)

    return tuple(
        (
            position            ,
            facilities[position],
        )
        for position in rng.sample(range(len(facilities)), k=limit)
    )


def find_group_containing(
    groups   : list[tuple[int, ...]],
    facility : int                  ,
) -> tuple[int, ...]:
    for group in groups:
        if facility in group:
            return tuple(sorted(group))

    raise ValueError(f"Facility {facility} was not found in any max-k-cut group.")


def build_max_k_cut_local_replacement_pool(groups, removed_facility):
    try:
        removed_group = find_group_containing(groups, removed_facility)
    except ValueError as exc:
        return tuple(), tuple(), f"{exc} Max-k-cut local reoptimization will be marked infeasible."

    replacement_pool = tuple(
        value
        for value in removed_group
        if  value != removed_facility
    )

    if not replacement_pool:
        return removed_group, replacement_pool, (
            "The max-k-cut group that contains the removed facility has no replacement candidate after forbidding the removed facility."
        )

    return removed_group, replacement_pool, None


def stable_seed_from_values(*values) -> int:
    seed_text = "|".join(str(value) for value in values)
    seed      = int(GLOBAL_SEED)

    for index, char in enumerate(seed_text):
        seed += (index + 1) * ord(char)

    return seed


def replacement_candidate_universe(
    n_facilities        : int,
    removed_facility    : int,
    excluded_facilities : tuple[int, ...] | list[int] | set[int],
) -> tuple[int, ...]:
    excluded = {
        int(removed_facility),
        *{
            int(facility)
            for facility in excluded_facilities
        },
    }

    return tuple(
        facility
        for facility in range(int(n_facilities))
        if  facility not in excluded
    )


def build_random_replacement_pool_with_target_size(
    n_facilities        : int,
    removed_facility    : int,
    excluded_facilities : tuple[int, ...] | list[int] | set[int],
    target_size         : int,
    *,
    seed_values         : tuple[object, ...],
) -> tuple[tuple[int, ...], int]:
    candidates  = replacement_candidate_universe(n_facilities, removed_facility, excluded_facilities)
    sample_size = min(max(0, int(target_size)), len(candidates))

    if sample_size == 0:
        return tuple(), len(candidates)

    rng = random.Random(stable_seed_from_values(*seed_values))

    return tuple(sorted(rng.sample(candidates, k=sample_size))), len(candidates)


def build_nearest_replacement_pool_with_target_size(
    distances           : np.ndarray,
    removed_facility    : int,
    excluded_facilities : tuple[int, ...] | list[int] | set[int],
    target_size         : int,
) -> tuple[tuple[int, ...], int]:
    candidates = replacement_candidate_universe(
        distances.shape[0] ,
        removed_facility   ,
        excluded_facilities,
    )

    selected = tuple(
        sorted(
            candidates,
            key=lambda facility: (float(distances[int(removed_facility), facility]), facility),
        )[:max(0, int(target_size))]
    )

    return selected, len(candidates)


def build_pmedian_model(
    distances : np.ndarray,
    p         : int       ,
    *,
    allowed_facilities   =None,
    fixed_open_facilities=None,
    forbidden_facilities =None,
):
    if distances.ndim != 2 or distances.shape[0] != distances.shape[1]:
        raise ValueError("Distances must be a square 2D array.")

    n = distances.shape[0]

    fixed_set     = set() if fixed_open_facilities is None else {int(value) for value in fixed_open_facilities}
    forbidden_set = set() if forbidden_facilities  is None else {int(value) for value in forbidden_facilities }

    if not fixed_set.isdisjoint(forbidden_set):
        raise ValueError("A fixed-open facility cannot also be forbidden.")

    candidate_set = set(range(n))                   \
                    if   allowed_facilities is None \
                    else {
                        int(value)
                        for value in allowed_facilities
                    }

    candidate_set.difference_update(forbidden_set)
    candidate_set.update           (fixed_set    )

    candidate_facilities = sorted(candidate_set)

    if len(candidate_facilities) < p:
        raise ValueError(f"Candidate facility pool is too small: {len(candidate_facilities)} < {p}.")
    if len(fixed_set           ) > p:
        raise ValueError(f"Too many fixed facilities: {len(fixed_set)} > {p}.")

    model = Model(solver_name="CBC")
    model.verbose = 0

    y = [model.add_var(var_type=BINARY) for _ in candidate_facilities]
    x = []
    for _ in range(n):
        x_row = [
            model.add_var(var_type=BINARY)
            for _ in candidate_facilities
        ]

        x    .append    (x_row)
        model.add_constr(xsum(x_row) == 1)

        for pos in range(len(candidate_facilities)):
            model.add_constr(x_row[pos] <= y[pos])

    facility_to_pos = {
        facility : pos
        for pos, facility
        in enumerate(candidate_facilities)
    }

    for facility in fixed_set:
        if facility not in facility_to_pos:
            raise ValueError(f"Fixed facility {facility} is not available in the candidate pool.")

        model.add_constr(y[facility_to_pos[facility]] == 1)

    model.add_constr(xsum(y) == p)

    model.objective = minimize(
        xsum(
            float(distances[i, facility]) * x[i][pos]
            for i in range(n)
            for pos, facility in enumerate(candidate_facilities)
        )
    )

    return model, x, y, candidate_facilities


def optimize_model(
    model              : Model     ,
    time_limit_seconds : int | None,
):
    if time_limit_seconds is None:
        return model.optimize()

    return model.optimize(max_seconds=float(time_limit_seconds))


def solve_pmedian_variant(
    *,
    variant       : str,
    variant_order : int,
    distances     : np.ndarray,

    p : int,

    time_limit_seconds : int | None,
    allowed_facilities   =None,
    fixed_open_facilities=None,
    forbidden_facilities =None,
):
    n = distances.shape[0]

    fixed_set     = set()         if fixed_open_facilities is None else {int(value) for value in fixed_open_facilities}
    forbidden_set = set()         if forbidden_facilities  is None else {int(value) for value in forbidden_facilities }
    candidate_set = set(range(n)) if allowed_facilities    is None else {int(value) for value in allowed_facilities   }

    candidate_set.difference_update(forbidden_set)
    candidate_set.update           (fixed_set    )

    candidate_facilities = sorted(candidate_set)

    row = {
        "variant"       : variant      ,
        "variant_order" : variant_order,

        "candidate_facility_count"    : len(candidate_facilities),
        "fixed_facility_count"        : len(fixed_set           ),
        "forbidden_facility_count"    : len(forbidden_set       ),
        "candidate_facility_fraction" : len(candidate_facilities) / max(1, n),

        "fixed_facilities"     : format_facilities(fixed_set    ),
        "forbidden_facilities" : format_facilities(forbidden_set),

        "objective_value"       : np.nan,
        "model_build_seconds"   : np.nan,
        "solve_seconds"         : np.nan,
        "total_variant_seconds" : np.nan,

        "open_facilities_count" : 0 ,
        "open_facilities"       : "",

        "status"        : "ERROR",
        "error_message" : None   ,
    }

    if len(candidate_facilities) < p:
        row.update(
            {
                "status"        : "INFEASIBLE_POOL",
                "error_message" : f"Candidate facility pool smaller than p: {len(candidate_facilities)} < {p}."
            }
        )

        return row, tuple()

    if len(fixed_set) > p:
        row.update(
            {
                "status"        : "INVALID_FIXED_SET",
                "error_message" : f"Too many fixed facilities: {len(fixed_set)} > {p}."
            }
        )

        return row, tuple()

    build_started_at = perf_counter()

    try:
        model, _, y, candidate_facilities = build_pmedian_model(
            distances,
            p        ,
            allowed_facilities   =candidate_facilities,
            fixed_open_facilities=fixed_set           ,
            forbidden_facilities =forbidden_set       ,
        )

        build_seconds    = perf_counter() - build_started_at
        solve_started_at = perf_counter()

        status = optimize_model(model, time_limit_seconds)

        solve_seconds = perf_counter() - solve_started_at

        has_incumbent    = status in {
            OptimizationStatus.OPTIMAL ,
            OptimizationStatus.FEASIBLE,
        }

        solver_objective = finite_or_none(model.objective_value if has_incumbent else None)
        open_facilities  = tuple()

        validated_objective = None

        if has_incumbent:
            open_facilities = tuple(
                extract_open_facilities_candidates(candidate_facilities, y)
            )

            if len(open_facilities) != p:
                raise ValueError(f"Expected {p} open facilities, but recovered {len(open_facilities)}.")

            validated_objective = compute_solution_cost(distances, open_facilities)

            if solver_objective is not None and validated_objective is not None:
                if abs(solver_objective - validated_objective) > 1e-4:
                    raise ValueError(f"Solver objective and validated objective do not match: {solver_objective} vs {validated_objective}.")

        objective_value = validated_objective if validated_objective is not None else solver_objective

        row.update(
            {
                "objective_value" : objective_value if objective_value is not None else np.nan,

                "model_build_seconds"   : build_seconds,
                "solve_seconds"         : solve_seconds,
                "total_variant_seconds" : build_seconds + solve_seconds,

                "open_facilities_count" : len              (open_facilities),
                "open_facilities"       : format_facilities(open_facilities),

                "status"        : getattr(status, "name", str(status)),
                "error_message" : None,
            }
        )

        return row, as_sorted_tuple(open_facilities)
    except Exception as exc:
        row.update(
            {
                "status"              : "ERROR",
                "model_build_seconds" : perf_counter() - build_started_at,
                "error_message"       : f"{type(exc).__name__}: {exc}"   ,
            }
        )

        return row, tuple()
    finally:
        gc.collect()


def describe_solution_change(reference_open, candidate_open):
    reference_set = set(as_sorted_tuple(reference_open))
    candidate_set = set(as_sorted_tuple(candidate_open))

    overlap = sorted(reference_set.intersection(candidate_set))
    dropped = sorted(reference_set - candidate_set)
    added   = sorted(candidate_set - reference_set)

    return {
        "overlap_with_original_optimum_count"    : len(overlap),
        "overlap_with_original_optimum_fraction" : len(overlap) / max(1, len(reference_set)),

        "dropped_optimal_facilities_count" : len(dropped),
        "new_facilities_count"             : len(added  ),

        "dropped_optimal_facilities" : format_facilities(dropped),
        "new_facilities"             : format_facilities(added  ),
    }

### BENCHMARK EXECUTION

In [14]:
VARIANT_SPECS = [
    {"variant" : "baseline_all_facilities"        , "variant_order" : 1, "analysis_key" : None                            , "pool_strategy" : "all_facilities"             },
    {"variant" : "max_k_cut_original"             , "variant_order" : 2, "analysis_key" : "max_k_cut_original"            , "pool_strategy" : "max_k_cut_group"            },
    {"variant" : "max_k_cut_mst_trajectory"       , "variant_order" : 3, "analysis_key" : "max_k_cut_mst_trajectory"      , "pool_strategy" : "max_k_cut_group"            },
    {"variant" : "max_k_cut_pairwise_trajectory"  , "variant_order" : 4, "analysis_key" : "max_k_cut_pairwise_trajectory" , "pool_strategy" : "max_k_cut_group"            },
    {"variant" : "max_k_cut_mst_range_guard"      , "variant_order" : 5, "analysis_key" : "max_k_cut_mst_range_guard"     , "pool_strategy" : "max_k_cut_group"            },
    {"variant" : "max_k_cut_pairwise_range_guard" , "variant_order" : 6, "analysis_key" : "max_k_cut_pairwise_range_guard", "pool_strategy" : "max_k_cut_group"            },
    {"variant" : "random_original_pool_size"      , "variant_order" : 7, "analysis_key" : "max_k_cut_original"            , "pool_strategy" : "random_original_pool_size"  },
    {"variant" : "nearest_original_pool_size"     , "variant_order" : 8, "analysis_key" : "max_k_cut_original"            , "pool_strategy" : "nearest_original_pool_size" },
]


def reoptimization_instance_name(
    instance_name    : str,
    removal_position : int,
    removed_facility : int | None = None
) -> str:
    instance_id = Path(canonical_instance_name(instance_name)).stem

    if removed_facility is None or pd.isna(removed_facility):
        return f"{instance_id}__remove_pos_{removal_position}"

    return f"{instance_id}__remove_pos_{removal_position}__facility_{int(removed_facility) + 1}"


def planned_reoptimization_positions(p_value) -> range:
    try:
        p_int = int(p_value)
    except (TypeError, ValueError):
        p_int = 0

    scenario_count = max(1, min(p_int, MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE))

    return range(scenario_count)


def build_pipeline_error_rows(
    instance_name  ,
    instance_id    ,
    metadata       ,
    best_known_cost,
    error_message  ,
):
    rows = []

    for removal_position in planned_reoptimization_positions(metadata.get("p", np.nan)):
        reopt_name = reoptimization_instance_name(instance_name, removal_position)

        for spec in VARIANT_SPECS:
            rows.append(
                {
                    "instance"                : instance_name,
                    "reoptimization_instance" : reopt_name   ,
                    "instance_id"             : instance_id  ,

                    "n" : metadata.get("n", np.nan),
                    "p" : metadata.get("p", np.nan),

                    "best_known_cost" : best_known_cost if best_known_cost is not None else np.nan,

                    "removal_position"            : removal_position,
                    "removed_facility_position"   : removal_position,
                    "removed_facility"            : np.nan,
                    "removed_facility_zero_based" : np.nan,

                    "variant"       : spec["variant"      ],
                    "variant_order" : spec["variant_order"],

                    "status"        : "PIPELINE_ERROR",
                    "error_message" : error_message   ,
                }
            )

    return rows


def build_pipeline_error_structure_rows(
    instance_name   ,
    instance_id     ,
    metadata        ,
    best_known_cost ,
    error_message   ,
    pipeline_seconds,
):
    return [
        {
            "instance"                : instance_name,
            "instance_id"             : instance_id  ,
            "reoptimization_instance" : reoptimization_instance_name(instance_name, removal_position),

            "n" : metadata.get("n", np.nan),
            "p" : metadata.get("p", np.nan),

            "best_known_cost" : best_known_cost if best_known_cost is not None else np.nan,

            "removal_position"            : removal_position,
            "removed_facility_position"   : removal_position,
            "removed_facility"            : np.nan,
            "removed_facility_zero_based" : np.nan,

            "pipeline_seconds" : pipeline_seconds,
            "error_message"    : error_message   ,
        }
        for removal_position in planned_reoptimization_positions(metadata.get("p", np.nan))
    ]


def analysis_for_structure_row(analyses):
    row = {}

    for key, analysis in analyses.items():
        prefix = f"{key}__"

        for field in [
            "memory_strategy"                        ,
            "analysis_memory_size"                   ,
            "analysis_component_count"               ,
            "analysis_is_connected"                  ,
            "cooccurrence_edges"                     ,
            "cooccurrence_total_weight"              ,
            "max_k_cut_fraction_cut"                 ,
            "max_k_cut_best_facility_separation"     ,
            "pre_connection_component_count"         ,
            "post_connection_component_count"        ,
            "trajectory_added_solution_count"        ,
            "trajectory_movement_count"              ,
            "trajectory_connection_edge_count"       ,
            "trajectory_generated_out_of_range_count",
            "guard_fallback_used"                    ,
            "best_component_solution_count"          ,
        ]:
            row[prefix + field] = analysis.get(field, np.nan)

        row[prefix + "groups"] = format_groups(analysis.get("max_k_cut_groups", []))

    return row


def build_reoptimization_scenario_rows(
    *,
    instance_name                ,
    instance_id                  ,
    graph                        ,
    distances                    ,
    p                            ,
    best_known_cost              ,
    exact_row                    ,
    exact_open                   ,
    exact_objective              ,
    summary                      ,
    memory_analyses              ,
    memory_metadata              ,
    metaheuristic_runtime_seconds,
    structure_runtime_seconds    ,
    best_facilities              ,
    meta_best_set                ,
    removal_position             ,
    removed_facility             ,
):
    remaining_optimal = tuple(
        facility
        for facility in exact_open
        if  facility != removed_facility
    )

    reopt_name = reoptimization_instance_name(instance_name, removal_position, removed_facility)

    common = {
        "instance"                : instance_name,
        "reoptimization_instance" : reopt_name   ,
        "instance_id"             : instance_id  ,

        "n" : int(graph["n"]),
        "p" : p              ,

        "best_known_cost" : best_known_cost if best_known_cost is not None else np.nan,

        "exact_original_status"              : exact_row["status"               ],
        "exact_original_objective"           : exact_row["objective_value"      ],
        "exact_original_model_build_seconds" : exact_row["model_build_seconds"  ],
        "exact_original_solve_seconds"       : exact_row["solve_seconds"        ],
        "exact_original_total_seconds"       : exact_row["total_variant_seconds"],

        "exact_original_open_facilities"           : format_facilities       (exact_open),
        "exact_original_gap_to_best_known_percent" : gap_to_reference_percent(exact_objective, best_known_cost),

        "reference_source"     : "exact_optimum",
        "reference_cost"       : exact_row["objective_value"] ,
        "reference_facilities" : format_facilities(exact_open),

        "metaheuristic_best_cost"                 : float(summary["tspmed_cost"]),
        "metaheuristic_runtime_seconds"           : metaheuristic_runtime_seconds,
        "metaheuristic_gap_to_best_known_percent" : gap_to_reference_percent(float(summary["tspmed_cost"]), best_known_cost),
        "metaheuristic_gap_to_exact_percent"      : gap_to_reference_percent(float(summary["tspmed_cost"]), exact_objective),

        "metaheuristic_best_facilities"           :               format_facilities(best_facilities),
        "metaheuristic_exact_overlap_count"       : len(set(exact_open).intersection(meta_best_set)),
        "metaheuristic_exact_overlap_fraction"    : len(set(exact_open).intersection(meta_best_set)) / max(1, p),

        "structure_extraction_runtime_seconds" : structure_runtime_seconds,
        "pre_reoptimization_seconds"           : exact_row["total_variant_seconds"] + metaheuristic_runtime_seconds + structure_runtime_seconds,

        "removed_facility"                   : removed_facility + 1,
        "removed_facility_zero_based"        : removed_facility,
        "removal_position"                   : removal_position,
        "removed_facility_position"          : removal_position,
        "remaining_optimal_facilities"       : format_facilities(remaining_optimal),
        "remaining_optimal_facilities_count" : len              (remaining_optimal),
        "removed_facility_in_meta_best"      : int(removed_facility in meta_best_set),

        **memory_metadata,
    }

    structure_row = {
        **common,

        "error_message" : None,

        **analysis_for_structure_row(memory_analyses)
    }

    original_analysis = memory_analyses["max_k_cut_original"]
    original_removed_group, original_replacement_pool, original_group_error = build_max_k_cut_local_replacement_pool(
        original_analysis["max_k_cut_groups"], removed_facility,
    )
    original_replacement_pool_size = len(original_replacement_pool)

    rows = []

    for spec in VARIANT_SPECS:
        allowed_facilities = None
        variant_analysis   = None
        group_error        = None
        removed_group      = tuple()
        replacement_pool   = tuple()

        pool_strategy                  = spec.get("pool_strategy", "max_k_cut_group" if spec["analysis_key"] is not None else "all_facilities")
        pool_reference_variant         = None
        pool_target_size               = np.nan
        pool_available_candidate_count = np.nan
        pool_excluded_best_facilities  = tuple()

        if pool_strategy == "all_facilities":
            pool_reference_variant = "all_facilities_baseline"
        elif pool_strategy == "max_k_cut_group":
            variant_analysis = memory_analyses[spec["analysis_key"]]

            if spec["analysis_key"] == "max_k_cut_original":
                removed_group    = original_removed_group
                replacement_pool = original_replacement_pool
                group_error      = original_group_error
            else:
                removed_group, replacement_pool, group_error = build_max_k_cut_local_replacement_pool(
                    variant_analysis["max_k_cut_groups"], removed_facility,
                )

            allowed_facilities              = replacement_pool
            pool_reference_variant          = spec["analysis_key"]
            pool_target_size                = len(replacement_pool)
            pool_available_candidate_count  = len(replacement_pool)
        elif pool_strategy == "random_original_pool_size":
            variant_analysis = original_analysis

            removed_group = original_removed_group
            replacement_pool, pool_available_candidate_count = build_random_replacement_pool_with_target_size(
                int(graph["n"]),
                removed_facility,
                best_facilities ,
                original_replacement_pool_size,
                seed_values=(
                    GLOBAL_SEED,
                    instance_id,
                    removal_position,
                    removed_facility,
                    pool_strategy,
                ),
            )

            allowed_facilities             = replacement_pool
            pool_reference_variant         = "max_k_cut_original"
            pool_target_size               = original_replacement_pool_size
            pool_excluded_best_facilities  = best_facilities
        elif pool_strategy == "nearest_original_pool_size":
            variant_analysis = original_analysis

            removed_group = original_removed_group
            replacement_pool, pool_available_candidate_count = build_nearest_replacement_pool_with_target_size(
                distances,
                removed_facility,
                best_facilities ,
                original_replacement_pool_size,
            )

            allowed_facilities             = replacement_pool
            pool_reference_variant         = "max_k_cut_original"
            pool_target_size               = original_replacement_pool_size
            pool_excluded_best_facilities  = best_facilities
        else:
            raise ValueError(f"Unknown replacement pool strategy: {pool_strategy}")

        row, open_facilities = solve_pmedian_variant(
            variant      =spec["variant"      ],
            variant_order=spec["variant_order"],
            distances=distances,
            p        =p        ,

            time_limit_seconds=REOPT_TIME_LIMIT_SECONDS,
            allowed_facilities   =allowed_facilities  ,
            fixed_open_facilities=remaining_optimal   ,
            forbidden_facilities =(removed_facility,) ,
        )

        if group_error is not None:
            row["error_message"] = group_error

        row.update(common)
        row.update(
            {
                "replacement_pool_strategy"                  : pool_strategy,
                "replacement_pool_reference_variant"         : pool_reference_variant,
                "replacement_pool_target_size"               : pool_target_size,
                "replacement_pool_available_candidate_count" : pool_available_candidate_count,
                "replacement_pool_shortfall"                 : max(0, int(pool_target_size) - len(replacement_pool)) if pd.notna(pool_target_size) else np.nan,
                "replacement_pool_excluded_best_facilities"  : format_facilities(pool_excluded_best_facilities),
            }
        )
        if variant_analysis is None:
            row.update({"memory_strategy": "all_facilities_baseline"})
        else:
            row.update({key: value for key, value in variant_analysis.items() if key != "max_k_cut_groups"})
            row.update(
                {
                    "max_k_cut_removed_group_found"   : int(bool(removed_group)),
                    "max_k_cut_removed_group_size"    : len(removed_group   ),
                    "max_k_cut_replacement_pool_size" : len(replacement_pool),
                    "max_k_cut_removed_group"         : format_facilities(removed_group   ),
                    "max_k_cut_replacement_pool"      : format_facilities(replacement_pool),
                    "max_k_cut_groups_formatted"      : format_groups    (variant_analysis["max_k_cut_groups"]),
                }
            )

        row.update(describe_solution_change(exact_open, open_facilities))

        row["degradation_vs_exact_percent"] = gap_to_reference_percent(
            row["objective_value"] if pd.notna(row["objective_value"]) else None,
            exact_objective,
        )
        row["total_pipeline_seconds"] = common["pre_reoptimization_seconds"] + (
            row["total_variant_seconds"] if pd.notna(row["total_variant_seconds"]) else 0.0
        )

        rows.append(row)

    return rows, structure_row


def run_single_instance_analysis(instance_name):
    instance_name = canonical_instance_name(instance_name)
    instance_path = INSTANCES_DIR / instance_name
    instance_id   = Path(instance_name).stem

    metadata        = read_instance_metadata(instance_path)
    best_known_cost = BEST_KNOWN_COSTS.get  (instance_id  )

    started_at = perf_counter()

    try:
        graph     = read_orlibrary_graph    (instance_path     )
        distances = all_pairs_shortest_paths(graph["adjacency"])

        p = int(graph["p"])

        exact_row, exact_open = solve_pmedian_variant(
            variant      ="exact_reference",
            variant_order=0                ,
            distances=distances,
            p        =p        ,
            time_limit_seconds=EXACT_TIME_LIMIT_SECONDS,
        )

        if exact_row["status"] != "OPTIMAL":
            raise ValueError(f"The original exact model must be OPTIMAL. Received status={exact_row['status']}.")
        if len(exact_open) != p:
            raise ValueError(f"Expected exact solution with p={p} facilities, found {len(exact_open)}.")

        meta_started_at = perf_counter()

        summary, details = pymedian.solve_pmedian(
            instance_path,
            restarts      =RESTARTS      ,
            max_iter      =MAX_ITER      ,
            factor        =FACTOR        ,
            details_format=DETAILS_FORMAT,
        )

        metaheuristic_runtime_seconds = perf_counter() - meta_started_at

        structure_started_at = perf_counter()

        memory_analyses, memory_metadata = build_max_k_cut_memory_analyses(summary, details, distances)

        structure_runtime_seconds = perf_counter() - structure_started_at

        exact_objective = exact_row["objective_value"] if pd.notna(exact_row["objective_value"]) else None

        best_facilities = best_facilities_from_summary(summary)
        meta_best_set   = set(best_facilities)

        rows           = []
        structure_rows = []
        for removal_position, removed_facility in reoptimization_removals(
            exact_open, MAX_REOPTIMIZATION_SCENARIOS_PER_INSTANCE,
        ):
            scenario_rows, structure_row = build_reoptimization_scenario_rows(
                instance_name=instance_name,
                instance_id  =instance_id  ,
                graph    =graph    ,
                distances=distances,
                p        =p        ,

                best_known_cost=best_known_cost,
                exact_row      =exact_row      ,
                exact_open     =exact_open     ,
                exact_objective=exact_objective,

                summary        =summary,
                memory_analyses=memory_analyses,
                memory_metadata=memory_metadata,

                metaheuristic_runtime_seconds=metaheuristic_runtime_seconds,
                structure_runtime_seconds    =structure_runtime_seconds    ,

                best_facilities =best_facilities ,
                meta_best_set   =meta_best_set   ,
                removal_position=removal_position,
                removed_facility=removed_facility,
            )
            rows          .extend(scenario_rows)
            structure_rows.append(structure_row)

        return rows, structure_rows
    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"

        return (
            build_pipeline_error_rows          (instance_name, instance_id, metadata, best_known_cost, error_message),
            build_pipeline_error_structure_rows(
                instance_name  ,
                instance_id    ,
                metadata       ,
                best_known_cost,
                error_message  ,
                perf_counter() - started_at,
            ),
        )

In [15]:
def sort_by_instance(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "instance" not in df.columns:
        return df

    ordered = df.copy()

    ordered["instance_order"] = ordered["instance"].map(lambda value: instance_sort_key(value)[0])

    sort_columns = ["instance_order", "instance"]

    for column in ["removal_position", "variant_order"]:
        if column in ordered.columns:
            sort_columns.append(column)

    return (
        ordered.sort_values(sort_columns, kind="stable")
               .drop       (columns="instance_order"   )
               .reset_index(drop   =True)
    )


def run_benchmark(instance_names):
    rows           = []
    structure_rows = []
    for instance_name in tqdm(
        instance_names,
        total        =len(instance_names)                             ,
        desc         ="Connected max-k-cut facility-removal benchmark",
        dynamic_ncols=True,
    ):
        instance_rows, instance_structure_rows = run_single_instance_analysis(instance_name)

        rows          .extend(instance_rows          )
        structure_rows.extend(instance_structure_rows)

    results_df    = pd.DataFrame(rows          )
    structures_df = pd.DataFrame(structure_rows)

    if not results_df.empty:
        results_df["variant_order"] = pd.to_numeric(
            results_df["variant_order"], errors="coerce"
        )

        results_df = sort_by_instance(results_df)

    structures_df = sort_by_instance(structures_df)

    return results_df, structures_df


def add_baseline_reference(results_df: pd.DataFrame) -> pd.DataFrame:
    merged = results_df.copy()

    if merged.empty:
        return merged

    baseline_df = (
        merged.loc[
            merged["variant"] == "baseline_all_facilities",
            [
                "reoptimization_instance",
                "status"                 ,
                "objective_value"        ,
                "total_variant_seconds"  ,
                "total_pipeline_seconds" ,
            ],
        ]
        .rename(
            columns={
                "status"                 : "baseline_status"                ,
                "objective_value"        : "baseline_objective_value"       ,
                "total_variant_seconds"  : "baseline_total_variant_seconds" ,
                "total_pipeline_seconds" : "baseline_total_pipeline_seconds",
            }
        )
    )

    merged = merged.merge(
        baseline_df, on="reoptimization_instance", how="left"
    )

    merged["gap_to_all_facilities_baseline_percent"] = np.where(
         merged["objective_value"].notna() & merged["baseline_objective_value"].notna(),
        100.0 * (merged["objective_value"] - merged["baseline_objective_value"]) / merged["baseline_objective_value"],
        np.nan,
    )

    merged["time_delta_to_baseline_seconds"] = merged["total_variant_seconds"] - merged["baseline_total_variant_seconds"]
    merged["baseline_is_optimal"           ] = merged["baseline_status"      ].eq("OPTIMAL")

    return merged


def build_wide_summary(results_df: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return results_df.copy()

    instance_columns = [
        "reoptimization_instance", "instance", "instance_id", "n", "p", "best_known_cost",
        "exact_original_objective", "exact_original_total_seconds", "exact_original_open_facilities",
        "reference_source", "reference_cost", "reference_facilities",
        "metaheuristic_best_cost", "metaheuristic_gap_to_exact_percent", "metaheuristic_runtime_seconds",
        "source_ltm_size", "top_fraction", "top_solution_count", "top_cost_cutoff",
        "full_ltm_cost_min", "full_ltm_cost_max", "removal_position", "removed_facility",
        "removed_facility_zero_based", "remaining_optimal_facilities",
    ]
    metric_columns = [
        "status", "objective_value", "degradation_vs_exact_percent", "gap_to_all_facilities_baseline_percent",
        "candidate_facility_count", "candidate_facility_fraction", "total_variant_seconds", "total_pipeline_seconds",
        "memory_strategy", "analysis_memory_size", "analysis_component_count", "analysis_is_connected",
        "replacement_pool_strategy", "replacement_pool_target_size", "replacement_pool_available_candidate_count",
        "replacement_pool_shortfall", "max_k_cut_fraction_cut", "max_k_cut_replacement_pool_size",
        "trajectory_added_solution_count", "trajectory_movement_count", "trajectory_generated_out_of_range_count",
        "guard_fallback_used",
    ]

    working = results_df.copy()
    for column in [*instance_columns, *metric_columns]:
        if column not in working.columns:
            working[column] = np.nan

    base_df = working[instance_columns].drop_duplicates("reoptimization_instance")
    wide_df = working.pivot(index="reoptimization_instance", columns="variant", values=metric_columns)

    wide_df.columns = [f"{variant}__{metric}" for metric, variant in wide_df.columns]

    return base_df.merge(wide_df.reset_index(), on="reoptimization_instance", how="left")


def build_variant_aggregate(results_df: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    working = results_df.copy()

    for column in [
        "objective_value", "candidate_facility_fraction", "total_variant_seconds", "total_pipeline_seconds",
        "degradation_vs_exact_percent", "gap_to_all_facilities_baseline_percent", "time_delta_to_baseline_seconds",
        "analysis_memory_size", "analysis_component_count", "max_k_cut_replacement_pool_size",
        "replacement_pool_target_size", "replacement_pool_shortfall",
        "trajectory_added_solution_count", "trajectory_movement_count",
    ]:
        if column not in working.columns:
            working[column] = np.nan

    return (
        working.groupby(["variant_order", "variant"], dropna=False)
        .agg(
            source_instances                   =("instance"               , "nunique"),
            reoptimization_instances           =("reoptimization_instance", "nunique"),

            solved_reoptimization_instances    =("objective_value", lambda s: int( s.notna()      .sum())),
            optimal_reoptimization_instances   =("status"         , lambda s: int((s == "OPTIMAL").sum())),

            mean_candidate_fraction            =("candidate_facility_fraction"           , "mean"),
            mean_replacement_pool_size         =("max_k_cut_replacement_pool_size"       , "mean"),
            mean_replacement_pool_target_size  =("replacement_pool_target_size"          , "mean"),
            mean_replacement_pool_shortfall    =("replacement_pool_shortfall"            , "mean"),
            mean_analysis_memory_size          =("analysis_memory_size"                  , "mean"),
            mean_analysis_components           =("analysis_component_count"              , "mean"),
            mean_trajectory_added              =("trajectory_added_solution_count"       , "mean"),
            mean_trajectory_movements          =("trajectory_movement_count"             , "mean"),
            mean_total_variant_seconds         =("total_variant_seconds"                 , "mean"),
            mean_total_pipeline_seconds        =("total_pipeline_seconds"                , "mean"),
            mean_time_delta_to_baseline_seconds=("time_delta_to_baseline_seconds"        , "mean"),
            mean_degradation_vs_exact_pct      =("degradation_vs_exact_percent"          , "mean"),
            mean_gap_to_baseline_pct           =("gap_to_all_facilities_baseline_percent", "mean"),
        )
        .reset_index()
        .sort_values("variant_order")
        .reset_index(drop=True      )
    )

### APPLY

In [16]:
raw_results_df, structures_df = run_benchmark(INSTANCE_NAMES)

Connected max-k-cut facility-removal benchmark:   0%|          | 0/20 [00:00<?, ?it/s]

In [17]:
results_df           = add_baseline_reference (raw_results_df)
wide_summary_df      = build_wide_summary     (results_df)
variant_aggregate_df = build_variant_aggregate(results_df)

error_series = results_df.get("error_message", pd.Series(index=results_df.index, dtype=object))
failures_df  = results_df[error_series.notna()].copy()

In [18]:
column = structures_df["max_k_cut_original__analysis_component_count"]

print("Mínimo :", column.min   ())
print("Mediana:", column.median())
print("Máximo :", column.max   ())

Mínimo : 1.0
Mediana: 1.0
Máximo : 5.0


In [19]:
display(wide_summary_df[[
    "reoptimization_instance",
    "n",
    "p",

    "metaheuristic_gap_to_exact_percent",
    "reference_facilities"              ,
    "removed_facility_zero_based"       ,

    "baseline_all_facilities__gap_to_all_facilities_baseline_percent"      ,
    "max_k_cut_original__gap_to_all_facilities_baseline_percent"           ,
    "max_k_cut_mst_trajectory__gap_to_all_facilities_baseline_percent"     ,
    "max_k_cut_pairwise_trajectory__gap_to_all_facilities_baseline_percent"  ,
    "max_k_cut_mst_range_guard__gap_to_all_facilities_baseline_percent"      ,
    "max_k_cut_pairwise_range_guard__gap_to_all_facilities_baseline_percent" ,
    "random_original_pool_size__gap_to_all_facilities_baseline_percent"      ,
    "nearest_original_pool_size__gap_to_all_facilities_baseline_percent"     ,

    "baseline_all_facilities__candidate_facility_fraction"      ,
    "max_k_cut_original__candidate_facility_fraction"           ,
    "max_k_cut_mst_trajectory__candidate_facility_fraction"     ,
    "max_k_cut_pairwise_trajectory__candidate_facility_fraction"  ,
    "max_k_cut_mst_range_guard__candidate_facility_fraction"      ,
    "max_k_cut_pairwise_range_guard__candidate_facility_fraction" ,
    "random_original_pool_size__candidate_facility_fraction"      ,
    "nearest_original_pool_size__candidate_facility_fraction"     ,

    "baseline_all_facilities__total_variant_seconds"      ,
    "max_k_cut_original__total_variant_seconds"           ,
    "max_k_cut_mst_trajectory__total_variant_seconds"     ,
    "max_k_cut_pairwise_trajectory__total_variant_seconds"  ,
    "max_k_cut_mst_range_guard__total_variant_seconds"      ,
    "max_k_cut_pairwise_range_guard__total_variant_seconds" ,
    "random_original_pool_size__total_variant_seconds"      ,
    "nearest_original_pool_size__total_variant_seconds"     ,
]].head().round(4))

,reoptimization_instance,n,p,metaheuristic_gap_to_exact_percent,reference_facilities,removed_facility_zero_based,baseline_all_facilities__gap_to_all_facilities_baseline_percent,max_k_cut_original__gap_to_all_facilities_baseline_percent,max_k_cut_mst_trajectory__gap_to_all_facilities_baseline_percent,max_k_cut_pairwise_trajectory__gap_to_all_facilities_baseline_percent,max_k_cut_mst_range_guard__gap_to_all_facilities_baseline_percent,max_k_cut_pairwise_range_guard__gap_to_all_facilities_baseline_percent,random_original_pool_size__gap_to_all_facilities_baseline_percent,nearest_original_pool_size__gap_to_all_facilities_baseline_percent,baseline_all_facilities__candidate_facility_fraction,max_k_cut_original__candidate_facility_fraction,max_k_cut_mst_trajectory__candidate_facility_fraction,max_k_cut_pairwise_trajectory__candidate_facility_fraction,max_k_cut_mst_range_guard__candidate_facility_fraction,max_k_cut_pairwise_range_guard__candidate_facility_fraction,random_original_pool_size__candidate_facility_fraction,nearest_original_pool_size__candidate_facility_fraction,baseline_all_facilities__total_variant_seconds,max_k_cut_original__total_variant_seconds,max_k_cut_mst_trajectory__total_variant_seconds,max_k_cut_pairwise_trajectory__total_variant_seconds,max_k_cut_mst_range_guard__total_variant_seconds,max_k_cut_pairwise_range_guard__total_variant_seconds,random_original_pool_size__total_variant_seconds,nearest_original_pool_size__total_variant_seconds
0,pmed1__remove_pos_0__facility_7,100,5,0.0,6 12 64 90 98,6.0,0.0,1.995906,1.995906,1.995906,1.995906,1.995906,6.363016,0.0,0.99,0.26,0.26,0.26,0.26,0.26,0.26,0.26,0.555106,0.204142,0.203635,0.230075,0.211311,0.208534,0.213968,0.22295
1,pmed1__remove_pos_2__facility_65,100,5,0.0,6 12 64 90 98,64.0,0.0,0.0,0.0,0.0,0.0,0.0,1.076187,0.0,0.99,0.26,0.26,0.26,0.26,0.26,0.26,0.26,0.588444,0.211954,0.214151,0.220128,0.268765,0.249232,0.241362,0.238641
2,pmed1__remove_pos_4__facility_99,100,5,0.0,6 12 64 90 98,98.0,0.0,0.0,0.0,0.0,0.0,0.0,0.377942,0.0,0.99,0.19,0.19,0.19,0.19,0.19,0.19,0.19,0.608134,0.143809,0.138777,0.142351,0.142105,0.142698,0.147411,0.153785
3,pmed2__remove_pos_0__facility_6,100,10,0.0,5 7 11 36 40 44 57 66 94 98,5.0,0.0,0.0,0.0,0.0,0.0,0.0,4.270376,0.0,0.99,0.19,0.19,0.19,0.19,0.19,0.19,0.19,0.546793,0.09093,0.092039,0.089546,0.087359,0.08856,0.093605,0.086126
4,pmed2__remove_pos_1__facility_8,100,10,0.0,5 7 11 36 40 44 57 66 94 98,7.0,0.0,0.0,0.0,0.0,0.0,0.0,3.439024,0.0,0.99,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.563653,0.062182,0.063906,0.063075,0.063407,0.062704,0.063083,0.065758


In [20]:
display(variant_aggregate_df[[
    "variant",

    "source_instances"                ,
    "reoptimization_instances"        ,
    "solved_reoptimization_instances" ,
    "optimal_reoptimization_instances",

    "mean_candidate_fraction"     ,
    "mean_replacement_pool_size",
    "mean_analysis_memory_size" ,
    "mean_analysis_components" ,
    "mean_trajectory_added"    ,

    "mean_total_variant_seconds",
    "mean_gap_to_baseline_pct"  ,
]].round(2))

,variant,source_instances,reoptimization_instances,solved_reoptimization_instances,optimal_reoptimization_instances,mean_candidate_fraction,mean_replacement_pool_size,mean_analysis_memory_size,mean_analysis_components,mean_trajectory_added,mean_total_variant_seconds,mean_gap_to_baseline_pct
0,baseline_all_facilities,20,60,57,57,0.99,NaN,NaN,NaN,NaN,5.82,0.00
1,max_k_cut_original,20,60,56,56,0.21,14.00,17.84,2.0,0.00,1.01,0.94
2,max_k_cut_mst_trajectory,20,60,57,57,0.21,13.98,22.42,1.0,4.58,1.02,0.92
3,max_k_cut_pairwise_trajectory,20,60,56,56,0.21,14.09,31.79,1.0,13.95,1.03,0.89
4,max_k_cut_mst_range_guard,20,60,57,57,0.21,13.98,22.42,1.0,4.58,1.02,0.92
5,max_k_cut_pairwise_range_guard,20,60,56,56,0.21,14.09,31.79,1.0,13.95,1.02,0.89
6,random_original_pool_size,20,60,56,56,0.21,14.00,17.84,2.0,0.00,1.02,1.74
7,nearest_original_pool_size,20,60,56,56,0.21,14.00,17.84,2.0,0.00,1.04,0.26


In [21]:
if SAVE_RESULTS_CSV:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    results_df     .to_csv(RAW_RESULTS_CSV, index=False)
    structures_df  .to_csv(STRUCTURES_CSV , index=False)
    wide_summary_df.to_csv(SUMMARY_CSV    , index=False)

    print(f"Raw results saved to : {RAW_RESULTS_CSV}")
    print(f"Structures saved to  : {STRUCTURES_CSV }")
    print(f"Summary saved to     : {SUMMARY_CSV    }")

    if not failures_df.empty:
        failures_df.to_csv(FAILURES_CSV, index=False)

        print(f"Failures saved to : {FAILURES_CSV}")

Raw results saved to : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_connected_max_k_cut_raw.csv
Structures saved to  : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_connected_max_k_cut_structures.csv
Summary saved to     : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_connected_max_k_cut_summary.csv
Failures saved to : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/facility_removal_connected_max_k_cut_failures.csv
